# Pipeline de dos etapas — evaluación encadenada dentro de la CV + odontogramas de los 10 niños reservados
**Settings:** GPU T4, Internet ON. **Inputs:** salidas de `benchmark-segmentacion-cv-nino` (predicciones fuera de fold y pesos), `benchmark-icdas-v4` (pesos del clasificador), `base-icdas-v4` (manifiesto v4), `datasets-limpios-icdas-segmentacion` (corpus de segmentación) y `ahmed-full-dataset` (fotos originales).

**Parte 1 — evaluación encadenada (sin fuga).** Recortes de la base v4 cuya foto está en el corpus de segmentación y cuyo niño está en el mismo fold en ambas bases (908 con diente FDI). Para cada uno, en el fold k:
- **M** — recorte manual → clasificador del fold k (predicción fuera de fold del benchmark).
- **G** — recorte automático desde la **máscara de referencia** (segmentación perfecta) → clasificador del fold k. Aísla el efecto del estilo de recorte.
- **P** — recorte automático desde la **máscara predicha** por el segmentador del fold k (el niño fue prueba en ese fold) → clasificador del fold k. Es el pipeline completo. Emparejamiento por número FDI (**P-FDI**, exige identificar bien el diente) y por ubicación (**P-ubic.**, el diente predicho que más se superpone con el de referencia, sin importar el número).
- Los márgenes de los recortes automáticos se calibran con la mediana de los márgenes de los recortes manuales respecto del diente en la máscara.
- Segmentador: regla fijada = mayor Dice de validación (media de 5 folds) entre los modelos que lo registran. Clasificador: el de mayor macro-F1 de validación en el benchmark ICDAS v4 (regla fijada).

**Parte 2 — odontogramas (cualitativo, sin cifras).** 10 niños reservados, nunca usados: 5 fotos con retractor → segmentación (ensamble de los 5 folds) → recorte por diente → clasificación (ensamble de los 5 folds) → peor clase por diente entre vistas.

**Parte 3 (Version 2, 2-nov) — demostración descriptiva.** Mismos 10 niños y mismos modelos que la Parte 2: guarda la salida por diente y foto (probabilidades de cada fold), calcula una tabla descriptiva sin referencia (dientes evaluados, 0/A/B/C, consistencia entre vistas, acuerdo entre los 5 modelos de fold, tiempo) y arma la figura foto → máscara FDI → clase por diente. Salidas nuevas: `reserva_por_diente_foto.csv`, `reserva_tabla_descriptiva.csv`, `reserva_tabla.tex`, `figuras/P4_ejemplos_mascara_clase.pdf/png`.

In [ ]:
!pip -q install timm albumentations segmentation-models-pytorch
import os, glob, json, re, zipfile, math, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F, timm, cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2
import torchvision.models as tvm
import segmentation_models_pytorch as smp
from transformers import SegformerForSemanticSegmentation
from sklearn.metrics import f1_score, cohen_kappa_score, accuracy_score
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
IN, W, TMP = '/kaggle/input', '/kaggle/working', '/kaggle/tmp'
IMG, SZ, CLASSES = 224, 512, ['0', 'A', 'B', 'C']
FIG = f'{W}/figuras'; os.makedirs(FIG, exist_ok=True)
find = lambda n: (glob.glob(f'{IN}/**/{n}', recursive=True) or [None])[0]
def save(fig, name):
    fig.savefig(f'{FIG}/{name}.png', dpi=300, bbox_inches='tight'); fig.savefig(f'{FIG}/{name}.pdf', bbox_inches='tight'); plt.close(fig)
CLS_RES = os.path.dirname(find('oof_predicciones.csv'))
_meta = pd.DataFrame([json.load(open(f)) for f in glob.glob(f'{CLS_RES}/*/fold*_meta.json')])
_val = _meta.groupby('model').best_val_macro_f1.mean().sort_values(ascending=False); print(_val.round(4))
CLS_MODEL = _val.index[0]
if CLS_MODEL.startswith('yolo'): print('AVISO: gana YOLO en validación; el pipeline no lo soporta, se usa el siguiente'); CLS_MODEL = [m for m in _val.index if not m.startswith('yolo')][0]
print('Clasificador elegido por validación:', CLS_MODEL)
SEG_RES = os.path.dirname(os.path.dirname(glob.glob(f'{IN}/**/unetpp_r50/fold0_meta.json', recursive=True)[0]))
IMGDIR = [d for d in glob.glob(f'{IN}/**/Dataset/Images', recursive=True) if os.path.isdir(d)][0]
print(CLS_RES, SEG_RES, IMGDIR)

## Datos: base v4, corpus de segmentación y clases

In [ ]:
D = f'{TMP}/d'
if not os.path.exists(f'{D}/icdas_v4'): zipfile.ZipFile(find('icdas_v4.zip')).extractall(D)
if not os.path.exists(f'{D}/seg'): zipfile.ZipFile(find('fdi_segmentation_curated.zip')).extractall(f'{D}/seg')
man = pd.read_csv(f'{D}/icdas_v4/manifest.csv', dtype={'micdas': str})
seg = pd.read_csv(find('segmentation_manifest.csv')).set_index('image_id')
ROOT = os.path.dirname(os.path.dirname(glob.glob(f'{D}/seg/**/{seg.image_file.iloc[0]}', recursive=True)[0]))
LM = glob.glob(f'{D}/seg/**/labelmap.txt', recursive=True)[0]
NAMES, COLORS = [], []
for l in open(LM, encoding='utf-8'):
    l = l.strip()
    if not l or l.startswith('#'): continue
    p = l.split(':'); NAMES.append(p[0].strip()); COLORS.append(tuple(int(v) for v in p[1].split(',')) if len(p) > 1 and p[1] else None)
bg = [i for i, n in enumerate(NAMES) if n.lower() in ('background', 'fondo', 'bg')]
if bg and bg[0] != 0: i = bg[0]; NAMES.insert(0, NAMES.pop(i)); COLORS.insert(0, COLORS.pop(i))
NC = len(NAMES)
def to_fdi(n):
    m = re.search(r'\d+', n); u = int(m.group()) if m else None
    if u is None or not 1 <= u <= 32: return None
    return 10 + (9 - u) if u <= 8 else 20 + (u - 8) if u <= 16 else 30 + (25 - u) if u <= 24 else 40 + (u - 24)
FDI = [None] + [to_fdi(n) for n in NAMES[1:]]; IDX = {f: i for i, f in enumerate(FDI) if f}
def read_mask(rel):
    m = cv2.cvtColor(cv2.imread(os.path.join(ROOT, rel))[..., :3], cv2.COLOR_BGR2RGB); out = np.zeros(m.shape[:2], np.uint8)
    for i, c in enumerate(COLORS):
        if i and c is not None: out[(m == np.array(c, np.uint8)).all(-1)] = i
    return out
ev = man[man.seg_mismo_fold & man.fdi.notna()].copy(); ev['fdi'] = ev.fdi.astype(int); ev['y'] = ev.micdas.map({c: i for i, c in enumerate(CLASSES)})
print(len(ev), 'recortes en la evaluación encadenada |', ev.child_id.nunique(), 'niños |', ev.seg_image_id.nunique(), 'fotos'); print(ev.micdas.value_counts().loc[CLASSES].to_dict())

## Regla fijada para el segmentador: mayor Dice de validación

In [ ]:
sm = pd.DataFrame([json.load(open(f)) for f in glob.glob(f'{SEG_RES}/*/fold*_meta.json')])
sv = sm.dropna(subset=['best_val_dice']).groupby('model').best_val_dice.agg(['mean', 'std', 'count']).sort_values('mean', ascending=False) if 'best_val_dice' in sm else None
print(sv.round(4)); SEG = sv.index[0]; print('Segmentador elegido:', SEG, '(YOLO11m-seg no registra Dice de validación: queda fuera de la regla)')
PRED = {k: np.load(f'{SEG_RES}/{SEG}/fold{k}_preds.npz') for k in range(5)}

## Cajas por diente y márgenes

In [ ]:
def comp_box(lab, c, ref=None):
    """Caja (en coords. de la máscara) de la componente conexa de la clase c; si se da ref, la que más se superpone con ref."""
    m = (lab == c).astype(np.uint8)
    if not m.any(): return None
    n, cc, st, _ = cv2.connectedComponentsWithStats(m, 8)
    if ref is not None:
        q = int(np.argmax([((cc == j) & ref).sum() if j else -1 for j in range(n)]))
    else:
        q = 1 + int(np.argmax(st[1:, cv2.CC_STAT_AREA]))
    x, y, w, h = st[q, :4]; return np.array([x, y, x + w, y + h], float)
def to_orig(box, s):
    Ainv = cv2.invertAffineTransform(np.array([[s.a11, s.a12, s.a13], [s.a21, s.a22, s.a23]], np.float64))
    pts = np.array([[box[0], box[1], 1], [box[2], box[1], 1], [box[2], box[3], 1], [box[0], box[3], 1]]) @ Ainv.T
    return np.r_[pts.min(0), pts.max(0)]
GT, rows = {}, []
for r in ev.itertuples():
    s = seg.loc[r.seg_image_id]
    if r.seg_image_id not in GT: GT[r.seg_image_id] = read_mask(s.mask_file)
    g = GT[r.seg_image_id]; p = PRED[r.fold][str(r.seg_image_id)]
    c = IDX[r.fdi]; gb = comp_box(g, c); ref = (g == c)
    pf = comp_box(p, c, ref)                                         # mismo número FDI
    lab = np.bincount(p[ref], minlength=NC); lab[0] = 0                 # ubicación: la etiqueta predicha que más cubre al diente de referencia
    pl_c = int(lab.argmax()) if lab.sum() else None
    pl = comp_box(p, pl_c, ref) if pl_c else None
    cover = lab.max() / max(1, ref.sum()) if lab.sum() else 0.0
    rows.append(dict(crop_id=r.crop_id, gtb=to_orig(gb, s) if gb is not None else None, pfdib=to_orig(pf, s) if pf is not None else None,
                     plocb=to_orig(pl, s) if pl is not None else None, ploc_fdi=FDI[pl_c] if pl_c else None, ploc_cover=cover))
B = pd.DataFrame(rows).set_index('crop_id'); ev = ev.set_index('crop_id').join(B)
man_box = ev[['x0', 'y0', 'x1', 'y1']].values.astype(float); ok = ev.gtb.notna().values
gt_box = np.stack([b if b is not None else np.full(4, np.nan) for b in ev.gtb]); gw, gh = gt_box[:, 2] - gt_box[:, 0], gt_box[:, 3] - gt_box[:, 1]
MARG = np.nanmedian(np.c_[(gt_box[:, 0] - man_box[:, 0]) / gw, (gt_box[:, 1] - man_box[:, 1]) / gh, (man_box[:, 2] - gt_box[:, 2]) / gw, (man_box[:, 3] - gt_box[:, 3]) / gh][ok], 0)
print('márgenes medianos del recorte manual respecto del diente en la máscara (izq, arr, der, abj, en fracción del diente):', MARG.round(3))
def iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0, min(a[3], b[3]) - max(a[1], b[1])); i = ix * iy
    return i / ((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - i)
def expand(b):
    w, h = b[2] - b[0], b[3] - b[1]; return np.array([b[0] - MARG[0] * w, b[1] - MARG[1] * h, b[2] + MARG[2] * w, b[3] + MARG[3] * h])
ev['iou_gt_manual'] = [iou(expand(g), m) if g is not None else np.nan for g, m in zip(ev.gtb, man_box)]
ev['iou_pfdi_manual'] = [iou(expand(g), m) if g is not None else np.nan for g, m in zip(ev.pfdib, man_box)]
ev['iou_ploc_manual'] = [iou(expand(g), m) if g is not None else np.nan for g, m in zip(ev.plocb, man_box)]
print('IoU con la caja manual (mediana): G', round(ev.iou_gt_manual.median(), 3), '| P-FDI', round(ev.iou_pfdi_manual.median(), 3), '| P-ubic.', round(ev.iou_ploc_manual.median(), 3))
print('diente encontrado con su número FDI:', round(ev.pfdib.notna().mean(), 3), '| encontrado por ubicación:', round(ev.plocb.notna().mean(), 3),
      '| número correcto entre los ubicados:', round((ev.ploc_fdi == ev.fdi)[ev.plocb.notna()].mean(), 3))

## Clasificador (el de mayor macro-F1 de validación en el benchmark v4 — Swin-T —, modelo del fold del niño)

In [ ]:
HEAVY_BOOST  = {0: 1.5, 1: 1.0, 2: 0.7, 3: 1.8}   # ResNet50 v6 / ConvNeXt V2 v11 / ViT-S v1b / Swin-T v1 — en el sampler
HYBRID_BOOST = {0: 1.0, 1: 1.3, 2: 1.0, 3: 2.0}   # CoAtNet-0 v2 / MaxViT-T — como peso de la loss
RECIPES = {
 'resnet50': dict(src='tv', ckpt='resnet50.IMAGENET1K_V2', head='mlp_bn', lr=3e-4, wd=3e-4, bs=32, epochs=50, patience=15,
                  warmup=5, min_lr=1e-6, warm_style='e', freeze=0, loss='ce', ls=0.1, mix='mixup', mixup_a=0.2, clip=2.0,
                  boost=HEAVY_BOOST, boost_in='sampler', aug='heavy_ge'),
 'convnextv2_tiny': dict(src='timm', ckpt='convnextv2_tiny.fcmae_ft_in1k', head='ln_drop', drop=0.4, lr=1e-4, wd=5e-4, bs=16,
                  epochs=60, patience=15, warmup=8, min_lr=1e-7, warm_style='e', freeze=8, unfreeze_stages=3, unfreeze='resume',
                  loss='focal', gamma=2.0, ls=0.0, mix='cutmix_mixup', cutmix_p=0.5, mixup_a=0.2, clip=None,
                  boost=HEAVY_BOOST, boost_in='sampler', aug='heavy'),
 'vit_small': dict(src='timm', ckpt='vit_small_patch16_224.augreg_in1k', head='linear', lr=1e-4, wd=0.05, bs=32, epochs=50,
                  patience=15, warmup=5, min_lr=1e-7, warm_style='e', freeze=0, loss='ce', ls=0.1, mix='mixup', mixup_a=0.2,
                  clip=1.0, boost=HEAVY_BOOST, boost_in='sampler', aug='heavy'),
 'swin_tiny': dict(src='timm', ckpt='swin_tiny_patch4_window7_224.ms_in1k', head='linear', lr=1e-4, wd=0.05, bs=32, epochs=50,
                  patience=15, warmup=5, min_lr=1e-7, warm_style='e', freeze=0, loss='ce', ls=0.1, mix='mixup', mixup_a=0.2,
                  clip=1.0, boost=HEAVY_BOOST, boost_in='sampler', aug='heavy'),
 'coatnet_0': dict(src='timm', ckpt='coatnet_0_rw_224.sw_in1k', head='ln_mlp', lr=3e-4, wd=0.05, bs=32, epochs=45, patience=12,
                  warmup=5, min_lr=0.0, warm_style='e+1', freeze=6, unfreeze_stages=2, unfreeze='restart_half',
                  loss='focal', gamma=2.0, ls=0.05, mix='cutmix_mixup', cutmix_p=0.5, mixup_a=0.3, clip=1.0,
                  boost=HYBRID_BOOST, boost_in='loss', aug='hybrid', drop_last=True),
 'maxvit_tiny': dict(src='timm', ckpt='maxvit_tiny_tf_224.in1k', head='ln_mlp', lr=3e-4, wd=0.05, bs=32, epochs=45, patience=12,
                  warmup=5, min_lr=0.0, warm_style='e+1', freeze=6, unfreeze_stages=2, unfreeze='restart_half',
                  loss='focal', gamma=2.0, ls=0.05, mix='cutmix_mixup', cutmix_p=0.5, mixup_a=0.3, clip=1.0,
                  boost=HYBRID_BOOST, boost_in='loss', aug='hybrid', drop_last=True),
 'yolo11m_cls': dict(src='ultralytics', ckpt='yolo11m-cls.pt', epochs=100, imgsz=224, batch=16, recipe='Ultralytics defaults'),
}


class Net(nn.Module):
    def __init__(self, r):
        super().__init__()
        if r['src'] == 'tv':
            self.backbone = tvm.resnet50(weights=None); nf = self.backbone.fc.in_features
            self.backbone.fc = nn.Identity()
        else:
            self.backbone = timm.create_model(r['ckpt'], pretrained=False, num_classes=0); nf = self.backbone.num_features
        h = r['head']
        if h == 'mlp_bn':  self.head = nn.Sequential(nn.Linear(nf, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.5), nn.Linear(512, 4))
        elif h == 'ln_drop': self.head = nn.Sequential(nn.LayerNorm(nf, eps=1e-6), nn.Dropout(r['drop']), nn.Linear(nf, 4))
        elif h == 'ln_mlp':  self.head = nn.Sequential(nn.LayerNorm(nf), nn.Dropout(0.5), nn.Linear(nf, 512), nn.GELU(), nn.Dropout(0.2), nn.Linear(512, 4))
        else:                self.head = nn.Linear(nf, 4)
    def forward(self, x): return self.head(self.backbone(x))
    def freeze(self):
        for p in self.backbone.parameters(): p.requires_grad = False
    def unfreeze_last(self, n):
        for st in list(self.backbone.stages)[-n:]:
            for p in st.parameters(): p.requires_grad = True
        for nm in ('norm_pre', 'norm'):
            if hasattr(self.backbone, nm) and isinstance(getattr(self.backbone, nm), nn.Module):
                for p in getattr(self.backbone, nm).parameters(): p.requires_grad = True


_cache = {}
def load_cls(k, m=CLS_MODEL):
    if (m, k) in _cache: return _cache[(m, k)]
    meta = json.load(open(f'{CLS_RES}/{m}/fold{k}_meta.json')); net = Net(RECIPES[m])
    sd = torch.load(f'{CLS_RES}/{m}/fold{k}_weights_fp16.pt', map_location='cpu'); net.load_state_dict({a: v.float() for a, v in sd.items()})
    tf = A.Compose([A.Resize(IMG, IMG), A.Normalize(mean=meta['mean'], std=meta['std']), ToTensorV2()])
    _cache.clear(); _cache[(m, k)] = (net.to(DEV).eval(), tf); return _cache[(m, k)]
@torch.no_grad()
def classify(k, crops, bs=64):
    net, tf = load_cls(k); P = []
    for i in range(0, len(crops), bs):
        x = torch.stack([tf(image=c)['image'] for c in crops[i:i + bs]]).to(DEV); P.append(torch.softmax(net(x).float(), 1).cpu().numpy())
    return np.concatenate(P) if P else np.zeros((0, 4))
_photos = {}
def photo(rel):
    if rel not in _photos:
        if len(_photos) > 40: _photos.clear()
        _photos[rel] = cv2.cvtColor(cv2.imread(os.path.join(IMGDIR, rel)), cv2.COLOR_BGR2RGB)
    return _photos[rel]
def cut(rel, b):
    im = photo(rel); x0, y0 = int(max(0, round(b[0]))), int(max(0, round(b[1]))); x1, y1 = int(min(im.shape[1], round(b[2]))), int(min(im.shape[0], round(b[3])))
    return im[y0:y1, x0:x1] if x1 - x0 >= 16 and y1 - y0 >= 16 else None

In [ ]:
oof = pd.read_csv(f'{CLS_RES}/oof_predicciones.csv'); oof = oof[oof.model == CLS_MODEL].set_index('crop_id')
ev['pred_M'] = oof.pred.reindex(ev.index).values
for k in range(5):
    sub = ev[ev.fold == k]
    for cond in ('gt', 'pfdi', 'ploc'):
        ids, crops = [], []
        for cid, r in sub.iterrows():
            b = r[cond + 'b']
            if b is None or (isinstance(b, float) and np.isnan(b)): continue
            c = cut(r.ahmed_image, expand(b))
            if c is not None: ids.append(cid); crops.append(c)
        P = classify(k, crops)
        ev.loc[ids, f'pred_{cond}'] = P.argmax(1)
        for j, cl in enumerate(CLASSES): ev.loc[ids, f'p_{cond}_{cl}'] = P[:, j]
    print('fold', k, 'listo')
ev.drop(columns=['gtb', 'pfdib', 'plocb']).to_csv(f'{W}/pipeline_por_recorte.csv')

## Métricas (IC 95 % bootstrap por niño; diferencias pareadas contra M)
- Condiciones: **M** (manual), **G** (máscara de referencia), **P-FDI** solo dientes encontrados con su número, **P-FDI extremo a extremo** (un diente no encontrado cuenta como predicción «0»: el pipeline no avisa), **P-ubic.**
- Macro-F1, κ ponderado cuadrático, exactitud y sensibilidad/especificidad en los cortes ICDAS ≥ 1, ≥ 3 y ≥ 5; concordancia entre la predicción sobre el recorte manual y la del pipeline.

In [ ]:
CUTS = {'icdas1': 1, 'icdas3': 2, 'icdas5': 3}
def mets(y, p):
    out = dict(n=len(y), macro_f1=f1_score(y, p, average='macro', labels=[0, 1, 2, 3], zero_division=0), kappa_w=cohen_kappa_score(y, p, weights='quadratic'), acc=accuracy_score(y, p))
    for nm, c0 in CUTS.items():
        t, s = y >= c0, p >= c0; out[f'sens_{nm}'] = (t & s).sum() / max(1, t.sum()); out[f'spec_{nm}'] = (~t & ~s).sum() / max(1, (~t).sum())
    return out
ev['pred_pfdi_e2e'] = ev.pred_pfdi.fillna(0)
CONDS = {'M (manual)': 'pred_M', 'G (máscara de referencia)': 'pred_gt', 'P-FDI (encontrados)': 'pred_pfdi', 'P-FDI extremo a extremo': 'pred_pfdi_e2e', 'P-ubic.': 'pred_ploc'}
kids = ev.child_id.unique(); rng = np.random.default_rng(0); draws = [rng.choice(kids, len(kids)) for _ in range(1000)]
byk = {c: g for c, g in ev.groupby('child_id')}
rows = []
for nm, col in CONDS.items():
    d = ev[ev[col].notna()]; pt = mets(d.y.values, d[col].astype(int).values); bs = []
    for dr in draws:
        x = pd.concat([byk[c] for c in dr]); x = x[x[col].notna()]; xm = x.pred_M.notna()
        a = mets(x.y.values, x[col].astype(int).values); b = mets(x.y.values, x.pred_M.astype(int).values)
        bs.append({**a, **{f'd_{q}': a[q] - b[q] for q in ('macro_f1', 'sens_icdas3', 'sens_icdas1')}})
    bs = pd.DataFrame(bs)
    rows.append(dict(condicion=nm, **pt, **{f'{q}_lo': bs[q].quantile(.025) for q in ('macro_f1', 'kappa_w', 'sens_icdas3', 'spec_icdas3')},
                     **{f'{q}_hi': bs[q].quantile(.975) for q in ('macro_f1', 'kappa_w', 'sens_icdas3', 'spec_icdas3')},
                     **{f'{q}_vsM': (np.nan if col == 'pred_M' else f'{bs[q].mean():+.3f} ({bs[q].quantile(.025):+.3f} a {bs[q].quantile(.975):+.3f})') for q in ('d_macro_f1', 'd_sens_icdas3')}))
R = pd.DataFrame(rows); R.to_csv(f'{W}/pipeline_metricas.csv', index=False)
pd.set_option('display.width', 250); print(R[['condicion', 'n', 'macro_f1', 'macro_f1_lo', 'macro_f1_hi', 'kappa_w', 'sens_icdas1', 'spec_icdas1', 'sens_icdas3', 'spec_icdas3', 'sens_icdas5', 'spec_icdas5', 'd_macro_f1_vsM', 'd_sens_icdas3_vsM']].round(3).to_string())
both = ev.pred_pfdi.notna()
print('Concordancia predicción manual vs pipeline (P-FDI, encontrados): κw', round(cohen_kappa_score(ev.pred_M[both], ev.pred_pfdi[both].astype(int), weights='quadratic'), 3),
      '| acuerdo exacto', round((ev.pred_M[both] == ev.pred_pfdi[both]).mean(), 3))
det = ev.groupby('micdas').apply(lambda d: pd.Series(dict(n=len(d), encontrado_fdi=d.pfdi_found.mean() if 'pfdi_found' in d else d.pred_pfdi.notna().mean(),
                                                         encontrado_ubic=d.pred_ploc.notna().mean(), iou_mediana=d.iou_pfdi_manual.median()))).loc[CLASSES]
print('Detección por clase:'); print(det.round(3).to_string()); det.to_csv(f'{W}/pipeline_deteccion_por_clase.csv')
print('Por vista (P-FDI extremo a extremo):'); print(ev.groupby('view').apply(lambda d: pd.Series(mets(d.y.values, d.pred_pfdi_e2e.astype(int).values))).round(3)[['n', 'macro_f1', 'sens_icdas3', 'spec_icdas3']].to_string())

## Ejemplos: recorte manual vs recorte del pipeline

In [ ]:
ex = ev[ev.pred_pfdi.notna()]
pick = pd.concat([ex[(ex.y == i) & (ex.pred_pfdi == i)].head(2) for i in range(4)] + [ex[ex.pred_pfdi != ex.pred_M].head(4)])
fig, axs = plt.subplots(2, len(pick), figsize=(1.7 * len(pick), 3.8), squeeze=False)
for j, (cid, r) in enumerate(pick.iterrows()):
    a = cut(r.ahmed_image, man_box[list(ev.index).index(cid)]); b = cut(r.ahmed_image, expand(B.loc[cid, 'pfdib']))
    for i, (im, lab) in enumerate(((a, f'manual → {CLASSES[int(r.pred_M)]}'), (b, f'pipeline → {CLASSES[int(r.pred_pfdi)]}'))):
        axs[i, j].imshow(cv2.resize(im, (160, 160))); axs[i, j].set_xticks([]); axs[i, j].set_yticks([]); axs[i, j].set_xlabel(lab, fontsize=7)
    axs[0, j].set_title(f'FDI {r.fdi} · ref {r.micdas}', fontsize=7)
fig.tight_layout(); save(fig, 'P1_ejemplos_manual_vs_pipeline')

## Parte 2 — odontogramas de los 10 niños reservados (cualitativo)

In [ ]:
YSZ = 640
SEG_RECIPES = {
 'unetpp_r50':     dict(kind='smp', arch='UnetPlusPlus', encoder='resnet50', lr=3e-4, enc_lr_mult=0.33, wd=1e-4, bs=8, epochs=80, patience=20, warmup=3),
 'deeplabv3p_r50': dict(kind='smp', arch='DeepLabV3Plus', encoder='resnet50', lr=3e-4, enc_lr_mult=0.33, wd=1e-4, bs=8, epochs=80, patience=20, warmup=3),
 'segformer_b2':   dict(kind='hf', ckpt='nvidia/mit-b2', lr=6e-5, enc_lr_mult=1.0, head_lr_mult=10.0, wd=0.01, bs=8, epochs=80, patience=20, warmup=3),
 'yolo11m_seg':    dict(kind='yolo', ckpt='yolo11m-seg.pt', epochs=150, patience=40, bs=16, lr0=0.01, imgsz=YSZ),
}
# Loss de los semánticos: CE + Dice (multiclase). AdamW, warmup lineal + coseno, AMP. YOLO: receta por defecto de Ultralytics salvo flips.
NC_ = NC
MEAN, STD = np.array([0.485, 0.456, 0.406], np.float32), np.array([0.229, 0.224, 0.225], np.float32)
class HFSeg(nn.Module):
    def __init__(self, ckpt):
        super().__init__(); self.m = SegformerForSemanticSegmentation.from_pretrained(ckpt, num_labels=NC, ignore_mismatched_sizes=True)
    def forward(self, x): return F.interpolate(self.m(pixel_values=x).logits, size=x.shape[-2:], mode='bilinear', align_corners=False)
def build(r, pretrained=True):
    if r['kind'] == 'smp':
        return getattr(smp, r['arch'])(encoder_name=r['encoder'], encoder_weights='imagenet' if pretrained else None, classes=NC)
    return HFSeg(r['ckpt'])

RESERVA = {"C0223": "003_007_734;003_007_735", "C0293": "003_007_988;003_007_989", "C0187": "003_007_602;003_007_603", "C0022": "003_007_1077;003_007_1078", "C0302": "003_008_1047;003_008_1048", "C0036": "003_007_1141;003_007_1142", "C0253": "003_007_841;003_007_842", "C0376": "003_008_1345;003_008_1346", "C0017": "003_007_1057;003_007_1058", "C0203": "003_007_660;003_007_661"}
allimgs = glob.glob(f'{IMGDIR}/retractors/*/*.jpg')
def norm(s): return s.replace('-', '_')
photos = {c: sorted({p for p in allimgs for k in ks.split(';') if f'_{k}_' in norm(os.path.basename(p))}) for c, ks in RESERVA.items()}
print({c: len(v) for c, v in photos.items()})
# letterbox igual al corpus: escala y desplazamiento estimados de las afines del corpus
sz = []
for iid, s in seg.sample(60, random_state=0).iterrows():
    im = cv2.imread(os.path.join(IMGDIR, s.ahmed_image)); h, w = im.shape[:2]; sz.append((s.a11 * max(w, h), s.a13 - (576 - s.a11 * w) / 2, s.a23 - (576 - s.a22 * h) / 2))
sz = np.array(sz); TGT = float(np.median(sz[:, 0])); print('lado mayor escalado (mediana):', round(TGT, 1), '| residuo de centrado x/y (mediana):', np.median(sz[:, 1:], 0).round(1))
c0 = cv2.imread(os.path.join(ROOT, seg.image_file.iloc[0])); PAD = tuple(int(v) for v in c0[0, 0])
def letterbox(im):
    h, w = im.shape[:2]; s = TGT / max(h, w); nw, nh = int(round(w * s)), int(round(h * s)); ox, oy = (576 - nw) / 2, (576 - nh) / 2
    out = np.full((576, 576, 3), PAD[::-1], np.uint8); out[int(oy):int(oy) + nh, int(ox):int(ox) + nw] = cv2.resize(im, (nw, nh), interpolation=cv2.INTER_AREA)
    return out, np.array([[s, 0, int(ox)], [0, s, int(oy)]], float)
segnets = []
for k in range(5):
    net = build(SEG_RECIPES[SEG], pretrained=False) if SEG_RECIPES[SEG]['kind'] == 'smp' else build(SEG_RECIPES[SEG])
    sd = torch.load(f'{SEG_RES}/{SEG}/fold{k}_weights_fp16.pt', map_location='cpu'); net.load_state_dict({a: v.float() for a, v in sd.items()}); segnets.append(net.to(DEV).eval())
@torch.no_grad()
def segment(im576):
    x = torch.from_numpy(((cv2.resize(im576, (SZ, SZ)).astype(np.float32) / 255 - MEAN) / STD).transpose(2, 0, 1))[None].to(DEV)
    pr = sum(torch.softmax(n(x).float(), 1) for n in segnets) / len(segnets)
    return F.interpolate(pr, size=(576, 576), mode='bilinear', align_corners=False)[0].argmax(0).byte().cpu().numpy()
SEV = {'0': 0, 'A': 1, 'B': 2, 'C': 3}; odo, overlays = {}, {}
for c, ps in photos.items():
    teeth = {}
    for p in ps:
        im = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB); lb, Aff = letterbox(im); lab = segment(lb); boxes, crops = [], []
        for ci in np.unique(lab):
            if ci == 0 or FDI[ci] is None or (lab == ci).sum() < 300: continue
            b = comp_box(lab, ci); s = pd.Series(dict(a11=Aff[0, 0], a12=0.0, a13=Aff[0, 2], a21=0.0, a22=Aff[1, 1], a23=Aff[1, 2]))
            bo = expand(to_orig(b, s)); x0, y0, x1, y1 = [int(v) for v in (max(0, bo[0]), max(0, bo[1]), min(im.shape[1], bo[2]), min(im.shape[0], bo[3]))]
            if x1 - x0 < 16 or y1 - y0 < 16: continue
            boxes.append((FDI[ci], (x0, y0, x1, y1))); crops.append(im[y0:y1, x0:x1])
        P = np.mean([classify(k, crops) for k in range(5)], 0) if crops else np.zeros((0, 4))
        preds = [CLASSES[i] for i in P.argmax(1)] if len(P) else []
        for (f, b), q in zip(boxes, preds):
            if f not in teeth or SEV[q] > SEV[teeth[f]]: teeth[f] = q
        overlays[(c, os.path.basename(p))] = (im, list(zip(boxes, preds)))
    odo[c] = teeth
pd.DataFrame([dict(child_id=c, fdi=f, clase=q) for c, t in odo.items() for f, q in sorted(t.items())]).to_csv(f'{W}/odontogramas_reserva.csv', index=False)
print({c: dict(pd.Series(t).value_counts()) for c, t in odo.items()})

In [ ]:
COL = {'0': '#d9ead3', 'A': '#ffe599', 'B': '#f6b26b', 'C': '#cc4125', None: '#eeeeee'}
UP = [18, 17, 16, 15, 14, 13, 12, 11, 21, 22, 23, 24, 25, 26, 27, 28]; LO = [48, 47, 46, 45, 44, 43, 42, 41, 31, 32, 33, 34, 35, 36, 37, 38]
fig, axs = plt.subplots(len(odo), 1, figsize=(8, 1.15 * len(odo)), squeeze=False)
for ax, (c, t) in zip(axs[:, 0], odo.items()):
    for row, arch in enumerate((UP, LO)):
        for j, f in enumerate(arch):
            q = t.get(f); ax.add_patch(Rectangle((j, 1 - row), 0.95, 0.9, color=COL[q])); ax.text(j + .47, 1.45 - row, str(f), ha='center', va='center', fontsize=5)
            if q: ax.text(j + .47, 1.2 - row, q, ha='center', va='center', fontsize=6, weight='bold')
    ax.set_xlim(0, 16); ax.set_ylim(0, 2); ax.axis('off'); ax.set_title(c, fontsize=8, loc='left')
fig.legend(handles=[Rectangle((0, 0), 1, 1, color=COL[k]) for k in ['0', 'A', 'B', 'C', None]], labels=['0', 'A', 'B', 'C', 'not seen'], ncol=5, loc='lower center', fontsize=7, frameon=False)
fig.tight_layout(rect=(0, .03, 1, 1)); save(fig, 'P2_odontogramas_reserva')
for c in list(odo)[:3]:
    keys = [k for k in overlays if k[0] == c][:5]
    if not keys: continue
    fig, axs = plt.subplots(1, len(keys), figsize=(4 * len(keys), 3.2), squeeze=False)
    for ax, k in zip(axs[0], keys):
        im, bp = overlays[k]; ax.imshow(im)
        for (f, (x0, y0, x1, y1)), q in bp:
            ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, lw=1.5, color={'0': '#6aa84f', 'A': '#f1c232', 'B': '#e69138', 'C': '#cc0000'}[q]))
            ax.text(x0, y0 - 8, f'{f}:{q}', fontsize=6, color='white', bbox=dict(facecolor='black', alpha=.5, pad=1, lw=0))
        ax.axis('off'); ax.set_title(k[1].split('_')[-2] if '_' in k[1] else k[1], fontsize=7)
    fig.tight_layout(); save(fig, f'P3_fotos_{c}')
print(sorted(os.listdir(FIG)))

## Parte 3 — demostración descriptiva: tabla por niño y figura de ejemplos (sin referencia; no se calcula exactitud)


In [ ]:
# ===== Parte 3a — salida por diente de los 10 niños reservados (ensamble 5 folds; guarda probabilidades por fold) =====
import time, copy
VIEWS_EN = {'frontal': 'Frontal', 'front': 'Frontal', 'left': 'Left lateral', 'right': 'Right lateral',
            'upper': 'Maxillary occlusal', 'maxillary': 'Maxillary occlusal', 'max': 'Maxillary occlusal',
            'lower': 'Mandibular occlusal', 'mandibular': 'Mandibular occlusal', 'mand': 'Mandibular occlusal'}
def view_of(p):
    folder = os.path.basename(os.path.dirname(p)).lower()
    for k, v in VIEWS_EN.items():
        if k in folder: return v
    return folder
print('carpetas de vista encontradas:', sorted({os.path.basename(os.path.dirname(p)) for ps in photos.values() for p in ps}))

def largest(lab, ci):
    m = (lab == ci).astype(np.uint8); n, cc, st, _ = cv2.connectedComponentsWithStats(m, 8)
    return cc == 1 + int(np.argmax(st[1:, cv2.CC_STAT_AREA])) if n > 1 else m.astype(bool)

rows3, store, t_gpu = [], {}, {}
for c, ps in photos.items():
    if DEV == 'cuda': torch.cuda.synchronize()
    t0 = time.perf_counter()
    for p in ps:
        im = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB); lb, Aff = letterbox(im); lab = segment(lb)
        s = pd.Series(dict(a11=Aff[0, 0], a12=0.0, a13=Aff[0, 2], a21=0.0, a22=Aff[1, 1], a23=Aff[1, 2]))
        teeth, crops = [], []
        for ci in np.unique(lab):                                   # mismos filtros que la Parte 2
            if ci == 0 or FDI[ci] is None or (lab == ci).sum() < 300: continue
            b = comp_box(lab, ci); bo = expand(to_orig(b, s))
            x0, y0, x1, y1 = [int(v) for v in (max(0, bo[0]), max(0, bo[1]), min(im.shape[1], bo[2]), min(im.shape[0], bo[3]))]
            if x1 - x0 < 16 or y1 - y0 < 16: continue
            teeth.append((ci, (x0, y0, x1, y1))); crops.append(im[y0:y1, x0:x1])
        Pk = np.stack([classify(k, crops) for k in range(5)]) if crops else np.zeros((5, 0, 4))   # (fold, diente, clase)
        Pm = Pk.mean(0)
        for j, (ci, box) in enumerate(teeth):
            votes = Pk[:, j].argmax(1)
            rows3.append(dict(child_id=c, photo=os.path.basename(p), view=view_of(p), fdi=FDI[ci], area576=int((lab == ci).sum()),
                              x0=box[0], y0=box[1], x1=box[2], y1=box[3], pred=CLASSES[int(Pm[j].argmax())], pmax=float(Pm[j].max()),
                              folds_agree=bool((votes == votes[0]).all()), n_fold_votes_for_pred=int((votes == Pm[j].argmax()).sum()),
                              **{f'p_{cl}': float(Pm[j, i]) for i, cl in enumerate(CLASSES)},
                              **{f'f{k}_pred': CLASSES[int(votes[k])] for k in range(5)}))
        store[(c, os.path.basename(p))] = dict(im=im, lab=lab, Aff=Aff, teeth=[(ci, box, CLASSES[int(Pm[j].argmax())]) for j, (ci, box) in enumerate(teeth)], view=view_of(p))
    if DEV == 'cuda': torch.cuda.synchronize()
    t_gpu[c] = time.perf_counter() - t0
T = pd.DataFrame(rows3); T.to_csv(f'{W}/reserva_por_diente_foto.csv', index=False)
# control: la peor clase por diente debe coincidir con los odontogramas de la Parte 2
worst = T.assign(sev=T.pred.map(SEV)).sort_values('sev').groupby(['child_id', 'fdi']).pred.last()
chk = all(worst.get((c, f)) == q for c, t in odo.items() for f, q in t.items()) and len(worst) == sum(len(t) for t in odo.values())
print('detecciones diente-foto:', len(T), '| coincide con los odontogramas de la Parte 2:', chk)


In [ ]:
# ===== Parte 3b — tiempo en CPU (1 hilo, un solo modelo por etapa: el de fold 0) y tabla descriptiva =====
torch.set_num_threads(1)
seg_cpu = copy.deepcopy(segnets[0]).cpu().eval()
net0, tf0 = load_cls(0); cls_cpu = copy.deepcopy(net0).cpu().eval()
@torch.no_grad()
def run_cpu(p):
    im = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB); lb, Aff = letterbox(im)
    x = torch.from_numpy(((cv2.resize(lb, (SZ, SZ)).astype(np.float32) / 255 - MEAN) / STD).transpose(2, 0, 1))[None]
    lab = F.interpolate(torch.softmax(seg_cpu(x).float(), 1), size=(576, 576), mode='bilinear', align_corners=False)[0].argmax(0).byte().numpy()
    s = pd.Series(dict(a11=Aff[0, 0], a12=0.0, a13=Aff[0, 2], a21=0.0, a22=Aff[1, 1], a23=Aff[1, 2])); crops = []
    for ci in np.unique(lab):
        if ci == 0 or FDI[ci] is None or (lab == ci).sum() < 300: continue
        bo = expand(to_orig(comp_box(lab, ci), s)); x0, y0, x1, y1 = [int(v) for v in (max(0, bo[0]), max(0, bo[1]), min(im.shape[1], bo[2]), min(im.shape[0], bo[3]))]
        if x1 - x0 >= 16 and y1 - y0 >= 16: crops.append(im[y0:y1, x0:x1])
    if crops: cls_cpu(torch.stack([tf0(image=cc)['image'] for cc in crops]))
    return len(crops)
t_cpu = {}
for c in list(photos)[:3]:                                     # 3 niños bastan para el tiempo
    t0 = time.perf_counter(); [run_cpu(p) for p in photos[c]]; t_cpu[c] = time.perf_counter() - t0
print('CPU 1 hilo, SegFormer-B2 + Swin-T (un modelo por etapa), segundos por niño (5 fotos):', {k: round(v, 1) for k, v in t_cpu.items()})
torch.set_num_threads(os.cpu_count())

rows = []
for c, d in T.groupby('child_id', sort=False):
    per_tooth = d.groupby('fdi').pred.agg(lambda s: sorted(set(s), key=SEV.get))
    w = d.assign(sev=d.pred.map(SEV)).sort_values('sev').groupby('fdi').pred.last()
    multi = d.groupby('fdi').photo.nunique() >= 2
    same = per_tooth[multi].map(len).eq(1)
    adj = per_tooth[multi].map(lambda v: SEV[v[-1]] - SEV[v[0]] <= 1)
    rows.append(dict(child=c, photos=d.photo.nunique(), teeth=len(w),
                     n0=(w == '0').sum(), nA=(w == 'A').sum(), nB=(w == 'B').sum(), nC=(w == 'C').sum(),
                     multi_view=int(multi.sum()), same_class=float(same.mean()) if multi.any() else np.nan,
                     within_one=float(adj.mean()) if multi.any() else np.nan,
                     folds_agree=float(d.folds_agree.mean()), gpu_s=t_gpu[c]))
TAB = pd.DataFrame(rows)
tot = dict(child='All', photos=TAB.photos.sum(), teeth=TAB.teeth.sum(), n0=TAB.n0.sum(), nA=TAB.nA.sum(), nB=TAB.nB.sum(), nC=TAB.nC.sum(),
           multi_view=TAB.multi_view.sum(), folds_agree=T.folds_agree.mean(), gpu_s=TAB.gpu_s.median())
# acuerdo entre vistas global = sobre todos los dientes vistos en >= 2 fotos (no promedio de niños)
allm = T.groupby(['child_id', 'fdi']).agg(nph=('photo', 'nunique'), cls=('pred', lambda s: sorted(set(s), key=SEV.get)))
allm = allm[allm.nph >= 2]
tot['same_class'] = allm.cls.map(len).eq(1).mean(); tot['within_one'] = allm.cls.map(lambda v: SEV[v[-1]] - SEV[v[0]] <= 1).mean()
TAB = pd.concat([TAB, pd.DataFrame([tot])], ignore_index=True); TAB.to_csv(f'{W}/reserva_tabla_descriptiva.csv', index=False)
print(TAB.round(3).to_string())
print('Detecciones diente-foto:', len(T), '| los 5 modelos de fold coinciden en', round(T.folds_agree.mean(), 3),
      '| probabilidad máxima del ensamble, mediana', round(T.pmax.median(), 3))
print('Dientes vistos en >= 2 fotos:', len(allm), '| misma clase en todas:', round(tot['same_class'], 3), '| a lo más una clase de diferencia:', round(tot['within_one'], 3))
# LaTeX listo para pegar
pct = lambda v: '--' if pd.isna(v) else f'{100 * v:.0f}'
TAB['label'] = [str(i + 1) if c != 'All' else 'All' for i, c in enumerate(TAB.child)]   # niños 1–10 en el paper (sin IDs internos)
lines = [f"{r.label} & {r.teeth} & {r.n0} & {r.nA} & {r.nB} & {r.nC} & {r.multi_view} & {pct(r.same_class)} & {pct(r.folds_agree)} \\\\" for r in TAB.itertuples()]
open(f'{W}/reserva_tabla.tex', 'w').write('\n'.join(lines)); print('\n'.join(lines))


In [ ]:
# ===== Parte 3c — figura: foto original | máscara FDI predicha | clase Merged-ICDAS por diente =====
# Para elegir a mano, llenar EXAMPLES con pares (child_id, nombre de archivo de la foto); vacío = selección automática.
EXAMPLES = []
N_ROWS = 4
CLS_COL = {'0': (106, 168, 79), 'A': (241, 194, 50), 'B': (230, 145, 56), 'C': (204, 0, 0)}
if not EXAMPLES:
    cand = T.groupby(['child_id', 'photo', 'view']).agg(nB=('pred', lambda s: (s == 'B').sum() + (s == 'C').sum()), nA=('pred', lambda s: (s == 'A').sum()),
                                                        n=('pred', 'size')).reset_index()
    cand = cand.sort_values(['nB', 'nA', 'n'], ascending=False)
    used_c, used_v = set(), set()
    for r in cand.itertuples():                              # una foto por vista, preferir niños distintos y fotos con B/C
        if r.view in used_v or r.child_id in used_c: continue
        EXAMPLES.append((r.child_id, r.photo)); used_v.add(r.view); used_c.add(r.child_id)
        if len(EXAMPLES) == N_ROWS: break
    for r in cand.itertuples():                              # si faltan filas, relajar la condición de niño distinto
        if len(EXAMPLES) == N_ROWS: break
        if r.view not in used_v: EXAMPLES.append((r.child_id, r.photo)); used_v.add(r.view)
# 10-nov-2026: la vista lateral de C0302 mostraba dientes temporales (no segmentados); se usa la lateral izquierda de C0376.
EXAMPLES = [e for e in EXAMPLES if 'Left' not in store[e]['view']] + [k for k in store if k[0] == 'C0376' and 'Left' in store[k]['view']][:1]
assert len(EXAMPLES) == 4, EXAMPLES
print('ejemplos:', [(c, p, store[(c, p)]['view']) for c, p in EXAMPLES])

def to_orig_mask(lab, Aff, shape):
    return cv2.warpAffine(lab, Aff, (shape[1], shape[0]), flags=cv2.INTER_NEAREST | cv2.WARP_INVERSE_MAP, borderValue=0)
def bbox_all(st, pad=0.04):
    xs = [b[0] for _, b, _ in st['teeth']] + [b[2] for _, b, _ in st['teeth']]; ys = [b[1] for _, b, _ in st['teeth']] + [b[3] for _, b, _ in st['teeth']]
    h, w = st['im'].shape[:2]; x0, x1, y0, y1 = min(xs), max(xs), min(ys), max(ys); px, py = pad * w, pad * h
    return int(max(0, x0 - px)), int(max(0, y0 - py)), int(min(w, x1 + px)), int(min(h, y1 + py))

# --- armado: cada fila con la altura que pide la proporción de su foto (sin franjas blancas) ---
COL_W = 4.0                                                  # ancho de cada panel en pulgadas
rows = []
for key in EXAMPLES:
    st = store[key]; im = st['im']; full = to_orig_mask(st['lab'], st['Aff'], im.shape)
    x0, y0, x1, y1 = bbox_all(st); rows.append((st, full, (x0, y0, x1, y1), (y1 - y0) / (x1 - x0)))
ratios = [r[3] for r in rows]
fig = plt.figure(figsize=(3 * COL_W + 0.5, COL_W * sum(ratios) + 0.9))
gs = fig.add_gridspec(len(rows), 3, height_ratios=ratios, hspace=0.06, wspace=0.03,
                      left=0.045, right=0.995, top=1 - 0.35 / fig.get_figheight(), bottom=0.5 / fig.get_figheight())
axs = np.array([[fig.add_subplot(gs[i, j]) for j in range(3)] for i in range(len(rows))])
for i, (st, full, (x0, y0, x1, y1), _) in enumerate(rows):
    im = st['im']; sub = im[y0:y1, x0:x1]; sw = x1 - x0
    over_fdi, over_cls = sub.astype(np.float32).copy(), sub.astype(np.float32).copy()
    texts_fdi, texts_cls = [], []
    for ci, box, q in st['teeth']:
        m = largest(full, ci)[y0:y1, x0:x1]
        if not m.any(): continue
        col = np.array(COLORS[ci] if COLORS[ci] is not None else (200, 200, 200), np.float32)
        over_fdi[m] = 0.45 * over_fdi[m] + 0.55 * col
        over_cls[m] = 0.45 * over_cls[m] + 0.55 * np.array(CLS_COL[q], np.float32)
        yy, xx = np.nonzero(m); cx, cy = xx.mean(), yy.mean()
        tw_pt = (xx.max() - xx.min() + 1) / sw * COL_W * 72  # ancho del diente en puntos tipográficos
        fs = float(np.clip(tw_pt * 0.42, 5.5, 8.5))          # etiqueta proporcional al diente
        texts_fdi.append((cx, cy, str(FDI[ci]), fs)); texts_cls.append((cx, cy, q, fs))
    for j, (img, txt) in enumerate(((sub, []), (over_fdi.astype(np.uint8), texts_fdi), (over_cls.astype(np.uint8), texts_cls))):
        ax = axs[i, j]; ax.imshow(img); ax.set_xticks([]); ax.set_yticks([])
        for cx, cy, t, fs in txt:
            ax.text(cx, cy, t, ha='center', va='center', fontsize=fs, weight='bold', color='white',
                    bbox=dict(facecolor='black', alpha=.45, pad=.5, lw=0))
    axs[i, 0].set_ylabel(st['view'].replace(' ', '\n'), fontsize=8.5, linespacing=1.1)
for j, t in enumerate(('Photograph', 'Stage 1: predicted tooth masks (FDI)', 'Stage 2: Merged-ICDAS class per tooth')):
    axs[0, j].set_title(t, fontsize=10)
fig.legend(handles=[Rectangle((0, 0), 1, 1, color=np.array(CLS_COL[k]) / 255) for k in CLASSES],
           labels=['0 (sound)', 'A (ICDAS 1–2)', 'B (ICDAS 3–4)', 'C (ICDAS 5–6)'], ncol=4, loc='lower center', bbox_to_anchor=(0.5, 0.0), fontsize=9, frameon=False)
save(fig, 'P4_ejemplos_mascara_clase')
print(sorted(os.listdir(FIG)))

In [ ]:
# ===== Parte 3d — ejemplo con las cuatro clases (0/A/B/C) =====
# En los 10 niños reservados no aparece ninguna C. Se busca una foto de la evaluación encadenada (niños de la CV, 166 fotos)
# procesada SOLO con los modelos del fold en que ese niño fue prueba en ambas etapas (máscara fuera de fold + clasificador del fold): sin fuga.
# Criterio fijado: que el pipeline prediga 0, A, B y C en la misma foto; entre esas, más C que coinciden con la referencia y luego mayor acuerdo con la referencia.
EXAMPLE_CV = None          # para fijar a mano: seg_image_id de la lista impresa abajo
cands = []
for k in range(5):                                            # por fold, para cargar cada clasificador una sola vez
    for sid, g in ev[ev.fold == k].groupby('seg_image_id'):
        s = seg.loc[sid]; lab = PRED[k][str(sid)]; im = photo(s.ahmed_image)
        Aff = np.array([[s.a11, s.a12, s.a13], [s.a21, s.a22, s.a23]], float)
        teeth, crops = [], []
        for ci in np.unique(lab):                               # mismos filtros que la Parte 2/3
            if ci == 0 or FDI[ci] is None or (lab == ci).sum() < 300: continue
            b = comp_box(lab, ci); bo = expand(to_orig(b, s))
            x0, y0, x1, y1 = [int(v) for v in (max(0, bo[0]), max(0, bo[1]), min(im.shape[1], bo[2]), min(im.shape[0], bo[3]))]
            if x1 - x0 < 16 or y1 - y0 < 16: continue
            teeth.append((ci, (x0, y0, x1, y1))); crops.append(im[y0:y1, x0:x1])
        if not crops: continue
        pred = [CLASSES[i] for i in classify(k, crops).argmax(1)]
        ref = dict(zip(g.fdi.astype(int), g.micdas))            # etiqueta de referencia de los dientes recortados a mano en esa foto
        both = [(FDI[ci], q) for (ci, _), q in zip(teeth, pred) if FDI[ci] in ref]
        cands.append(dict(seg_image_id=sid, child_id=g.child_id.iloc[0], fold=k, view=view_of(os.path.join(IMGDIR, s.ahmed_image)),
                          dientes=len(teeth), **{f'n{c}': pred.count(c) for c in CLASSES},
                          con_ref=len(both), acuerdo_ref=float(np.mean([ref[f] == q for f, q in both])) if both else np.nan,
                          C_ok=sum(ref[f] == 'C' and q == 'C' for f, q in both), ref_C=sum(v == 'C' for v in ref.values()),
                          st=dict(im=im, lab=lab, Aff=Aff, teeth=[(ci, b, q) for (ci, b), q in zip(teeth, pred)], view=view_of(os.path.join(IMGDIR, s.ahmed_image)),
                                  ref={f: ref[f] for f, _ in both})))
CV = pd.DataFrame(cands)
todas = CV[(CV[[f'n{c}' for c in CLASSES]] > 0).all(1)].sort_values(['C_ok', 'acuerdo_ref', 'nC'], ascending=False)
print(f'fotos de la CV con 0/A/B/C predichos: {len(todas)} de {len(CV)}')
print(todas.drop(columns='st').head(10).round(2).to_string(index=False))
pick = (CV[CV.seg_image_id == EXAMPLE_CV] if EXAMPLE_CV is not None else todas).iloc[0]
print('elegida:', pick.seg_image_id, pick.child_id, 'fold', pick.fold, pick['view'], '| acuerdo con la referencia en los dientes con etiqueta:',
      f"{pick.acuerdo_ref:.2f} ({pick.con_ref} dientes)", '| referencia de esos dientes:', pick.st['ref'])
CV.drop(columns='st').to_csv(f'{W}/cv_fotos_candidatas_0ABC.csv', index=False)

def draw_rows(sts, labels, name, note=None):
    rows = []
    for st in sts:
        full = to_orig_mask(st['lab'], st['Aff'], st['im'].shape); x0, y0, x1, y1 = bbox_all(st)
        rows.append((st, full, (x0, y0, x1, y1), (y1 - y0) / (x1 - x0)))
    ratios = [r[3] for r in rows]
    fig = plt.figure(figsize=(3 * COL_W + 0.5, COL_W * sum(ratios) + 0.9 + (0.25 if note else 0)))
    gs = fig.add_gridspec(len(rows), 3, height_ratios=ratios, hspace=0.06, wspace=0.03,
                          left=0.045, right=0.995, top=1 - 0.35 / fig.get_figheight(), bottom=(0.5 + (0.25 if note else 0)) / fig.get_figheight())
    for i, (st, full, (x0, y0, x1, y1), _) in enumerate(rows):
        sub = st['im'][y0:y1, x0:x1]; sw = x1 - x0
        over_fdi, over_cls = sub.astype(np.float32).copy(), sub.astype(np.float32).copy(); tf, tc = [], []
        for ci, box, q in st['teeth']:
            m = largest(full, ci)[y0:y1, x0:x1]
            if not m.any(): continue
            col = np.array(COLORS[ci] if COLORS[ci] is not None else (200, 200, 200), np.float32)
            over_fdi[m] = 0.45 * over_fdi[m] + 0.55 * col; over_cls[m] = 0.45 * over_cls[m] + 0.55 * np.array(CLS_COL[q], np.float32)
            yy, xx = np.nonzero(m); cx, cy = xx.mean(), yy.mean(); fs = float(np.clip((xx.max() - xx.min() + 1) / sw * COL_W * 72 * 0.42, 5.5, 8.5))
            tf.append((cx, cy, str(FDI[ci]), fs)); tc.append((cx, cy, q, fs))
        for j, (img, txt) in enumerate(((sub, []), (over_fdi.astype(np.uint8), tf), (over_cls.astype(np.uint8), tc))):
            ax = fig.add_subplot(gs[i, j]); ax.imshow(img); ax.set_xticks([]); ax.set_yticks([])
            for cx, cy, t, fs in txt:
                ax.text(cx, cy, t, ha='center', va='center', fontsize=fs, weight='bold', color='white', bbox=dict(facecolor='black', alpha=.45, pad=.5, lw=0))
            if j == 0: ax.set_ylabel(labels[i], fontsize=8.5, linespacing=1.1)
            if i == 0: ax.set_title(('Photograph', 'Stage 1: predicted tooth masks (FDI)', 'Stage 2: Merged-ICDAS class per tooth')[j], fontsize=10)
    fig.legend(handles=[Rectangle((0, 0), 1, 1, color=np.array(CLS_COL[k]) / 255) for k in CLASSES],
               labels=['0 (sound)', 'A (ICDAS 1–2)', 'B (ICDAS 3–4)', 'C (ICDAS 5–6)'], ncol=4, loc='lower center',
               bbox_to_anchor=(0.5, (0.25 if note else 0) / fig.get_figheight()), fontsize=9, frameon=False)
    if note: fig.text(0.5, 0.02 / fig.get_figheight() * 4, note, ha='center', va='bottom', fontsize=7.5, style='italic')
    save(fig, name)

lab_res = [store[k]['view'].replace(' ', '\n') for k in EXAMPLES]
draw_rows([store[k] for k in EXAMPLES] + [pick.st], lab_res + [pick['view'].replace(' ', '\n') + '\n(CV, out-of-fold)'], 'P4b_ejemplos_con_C',
          note='Rows 1–4: reserved children (five-fold ensemble). Last row: cross-validation child, models of the fold in which it was a test case.')
draw_rows([pick.st], [pick['view'].replace(' ', '\n')], 'P5_ejemplo_cv_0ABC')
print(sorted(os.listdir(FIG)))

In [ ]:
# ===== Parte 3e (10-nov-2026) — figura con un caso de dientes temporales =====
# agregar la lateral izquierda de C0302 (dientes temporales retenidos, que la etapa 1 no segmenta) como fila adicional.
lat302 = [k for k in store if k[0] == 'C0302' and 'Left' in store[k]['view']][:1]
assert len(lat302) == 1, lat302
rows_e = [store[k] for k in EXAMPLES] + [store[lat302[0]], pick.st]
labs_e = [store[k]['view'].replace(' ', '\n') for k in EXAMPLES] + ['Left lateral\n(retained\nprimary teeth)', pick['view'].replace(' ', '\n') + '\n(CV, out-of-fold)']
draw_rows(rows_e, labs_e, 'P4c_ejemplos_con_temporales',
          note='Rows 1–5: reserved children (five-fold ensemble); row 5 has retained primary teeth. Last row: cross-validation child, models of the fold in which it was a test case.')
print('P4c filas:', [(k[0], store[k]['view']) for k in EXAMPLES] + [(lat302[0][0], 'Left lateral')], '+ CV', pick.child_id)


In [ ]:
# ===== Parte 3f (11-nov-2026) — Additional file 6: los 10 niños reservados, 5 vistas, una página por niño =====
# reemplaza los odontogramas. Izquierda: foto; derecha: máscara predicha coloreada por clase Merged-ICDAS con su número FDI.
import io
from PIL import Image as PILImage
VIEW_ORDER = ['Frontal', 'Left lateral', 'Right lateral', 'Maxillary occlusal', 'Mandibular occlusal']
kids_order = [c for c in TAB.child if c != 'All']          # mismo orden que la Tabla 5 (niños 1-10)
pages = []
for n, cid in enumerate(kids_order, 1):
    keys = sorted([k for k in store if k[0] == cid], key=lambda k: VIEW_ORDER.index(store[k]['view']) if store[k]['view'] in VIEW_ORDER else 9)
    rows = []
    for k in keys:
        st = store[k]; full = to_orig_mask(st['lab'], st['Aff'], st['im'].shape); x0, y0, x1, y1 = bbox_all(st)
        rows.append((st, full, (x0, y0, x1, y1), (y1 - y0) / (x1 - x0)))
    fig = plt.figure(figsize=(8.27, 11.69))
    gs = fig.add_gridspec(len(rows), 2, height_ratios=[r[3] for r in rows], hspace=0.08, wspace=0.03, left=0.07, right=0.99, top=0.935, bottom=0.07)
    fig.suptitle(f'Reserved child {n}', fontsize=12, weight='bold', y=0.975)
    for i, (st, full, (x0, y0, x1, y1), _) in enumerate(rows):
        sub = st['im'][y0:y1, x0:x1]; over = sub.astype(np.float32).copy(); txt = []; sw = x1 - x0
        for ci, box, q in st['teeth']:
            m = largest(full, ci)[y0:y1, x0:x1]
            if not m.any(): continue
            over[m] = 0.45 * over[m] + 0.55 * np.array(CLS_COL[q], np.float32)
            yy, xx = np.nonzero(m); fs = float(np.clip((xx.max() - xx.min() + 1) / sw * 3.8 * 72 * 0.42, 5, 8))
            txt.append((xx.mean(), yy.mean(), str(FDI[ci]), fs))
        for j, img in enumerate((sub, over.astype(np.uint8))):
            ax = fig.add_subplot(gs[i, j]); ax.imshow(img); ax.set_xticks([]); ax.set_yticks([])
            if j == 1:
                for cx, cy, t, fs in txt:
                    ax.text(cx, cy, t, ha='center', va='center', fontsize=fs, weight='bold', color='white', bbox=dict(facecolor='black', alpha=.45, pad=.5, lw=0))
            else:
                ax.set_ylabel(st['view'].replace(' ', '\n'), fontsize=8.5, linespacing=1.1)
            if i == 0: ax.set_title(('Photograph', 'Predicted tooth masks (FDI number), coloured by Merged-ICDAS class')[j], fontsize=9)
    fig.legend(handles=[Rectangle((0, 0), 1, 1, color=np.array(CLS_COL[k]) / 255) for k in CLASSES],
               labels=['0 (sound)', 'A (ICDAS 1–2)', 'B (ICDAS 3–4)', 'C (ICDAS 5–6)'], ncol=4, loc='lower center', bbox_to_anchor=(0.5, 0.025), fontsize=9, frameon=False)
    fig.text(0.5, 0.008, 'Five-fold ensemble of each stage. Teeth without a mask were not segmented by stage 1.', ha='center', fontsize=7.5, style='italic')
    buf = io.BytesIO(); fig.savefig(buf, format='png', dpi=170); plt.close(fig); buf.seek(0)
    pages.append(PILImage.open(buf).convert('RGB'))
    print(n, cid, [store[k]['view'] for k in keys])
AF6 = f'{FIG}/Additional_file_6_reserved_children.pdf'
pages[0].save(AF6, save_all=True, append_images=pages[1:], resolution=170, quality=82)
print(AF6, os.path.getsize(AF6) // 1024, 'KB', len(pages), 'páginas')
